# 15 — Regulatory Researcher
> Paste in a regulation and get back every obligation, deadline, and penalty — each one cited to the exact article it came from.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


class RegulatoryObligation(BaseModel):
    source_article: str = Field(
        description="Exact article or section reference, e.g. 'Article 32(1)(a)'"
    )
    obligation: str = Field(description="What the regulated firm must do")
    applies_to: str = Field(description="Who this obligation applies to")
    is_ongoing: bool = Field(description="True if the obligation is continuous, not one-off")
    deadline: Optional[str] = Field(
        default=None, description="Specific deadline if stated in the regulation"
    )


class RegulatoryPenalty(BaseModel):
    source_article: str = Field(
        description="Exact article or section reference for this penalty"
    )
    trigger: str = Field(description="What violation triggers this penalty")
    maximum_fine: Optional[str] = Field(
        default=None, description="Maximum financial penalty as stated in the text"
    )
    other_consequences: List[str] = Field(
        description="Non-financial consequences (e.g. suspension, public censure)"
    )


class ComplianceSummary(BaseModel):
    regulation_name: str
    jurisdiction: str
    in_force_date: Optional[str] = Field(
        default=None, description="When the regulation takes effect"
    )
    obligations: List[RegulatoryObligation] = Field(
        description="All obligations found, each with exact article citation"
    )
    key_deadlines: List[str] = Field(
        description="Each deadline must include the article reference, e.g. 'Article 15: quarterly reporting within 30 days'"
    )
    penalties: List[RegulatoryPenalty] = Field(
        description="All penalties found, each with exact article citation"
    )
    high_priority_gaps: List[str] = Field(
        description="What firms most commonly miss or underestimate in compliance"
    )


RESEARCHER_SYSTEM = SystemMessage(
    """You are a regulatory compliance analyst specialising in extracting structured
compliance intelligence from regulatory texts.

Extract every obligation, deadline, and penalty from the regulation text provided.

CITATION RULE (mandatory):
  Every item in `obligations` and `penalties` MUST include the exact source_article
  (e.g. 'Article 5(1)(f)' or 'Section 12(3)'). If you cannot point to a specific
  article or section in the provided text, do not include the finding.
  Cite precisely; never paraphrase article numbers or invent references.

EXTRACTION RULES:
  - obligations  : what regulated parties must do (ongoing or one-off)
  - key_deadlines: combine article reference with the deadline text, e.g.
                   "Article 15(2): quarterly report within 30 days of quarter end"
  - penalties    : what happens when obligations are breached (cite source_article)
  - high_priority_gaps: what compliance teams most commonly overlook in practice

Do not speculate beyond what the text states."""
)


def run(regulation_text: str) -> ComplianceSummary:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    researcher = RESEARCHER_SYSTEM | llm.with_structured_output(ComplianceSummary)
    return researcher.invoke(
        HumanMessage(content="Regulation text to analyse:\n\n" + regulation_text)
    )

## The scenario

We're reviewing a fictional excerpt from the **EU Consumer Data Protection (Financial Services) Regulation 2025** — a regulation that binds payment institutions and e-money firms operating across the EU. It covers data retention windows, breach notification timelines, algorithmic decision disclosure, and the penalties for getting any of those wrong. The agent will extract every obligation and penalty, flag the exact article behind each one, and surface what firms most often overlook when trying to comply.

In [ ]:
REGULATION_TEXT = """
EU CONSUMER DATA PROTECTION (FINANCIAL SERVICES) REGULATION 2025 (CDPFS 2025)
Jurisdiction: European Union
In force: 1 March 2026

CHAPTER III: OBLIGATIONS ON PAYMENT INSTITUTIONS AND E-MONEY INSTITUTIONS

Article 8 -- Data Retention
8(1) A payment institution or e-money institution shall retain transaction records,
customer identification data, and supporting documentation for a minimum period of
seven years from the date of each transaction or the termination of the customer
relationship, whichever is later.
8(2) Retained data must be stored in a manner that ensures integrity, accessibility,
and confidentiality, and must be available to the competent authority upon request
within 72 hours.
8(3) Institutions must conduct an annual review of retained data to verify
compliance with this Article and document the outcome of that review.

Article 12 -- Breach Notification
12(1) Where an institution becomes aware of a personal data breach affecting
customer financial data, it shall notify the competent authority without undue
delay and, where feasible, within 24 hours of becoming aware of the breach.
12(2) If notification cannot be made within 24 hours, the institution shall
provide the notification as soon as possible and include the reasons for the delay.
12(3) Where the breach is likely to result in a high risk to the rights and
freedoms of affected customers, the institution shall also notify those customers
without undue delay.

Article 17 -- Algorithmic Decision-Making Disclosure
17(1) Where an institution uses an automated decision-making system to approve
or reject a payment or credit application, it shall inform the customer of this
fact at the time the decision is communicated.
17(2) Upon customer request, the institution shall provide a written explanation
of the logic and principal factors that influenced the automated decision within
15 business days.

CHAPTER VI: ENFORCEMENT

Article 31 -- Administrative Penalties
31(1) The competent authority may impose administrative penalties where it
determines that an institution has breached the obligations set out in this
Regulation. Financial penalties shall not exceed the higher of EUR 20,000,000
or 4% of the institution's total annual worldwide turnover in the preceding
financial year.
31(2) In determining the penalty, the competent authority shall take into account
the duration of the infringement, the degree of cooperation with the authority,
and whether the breach caused harm to customers.

Article 32 -- Temporary Prohibition
32(1) Where an institution repeatedly fails to comply with Article 8 or Article 12,
the competent authority may impose a temporary prohibition on the processing of new
customer data for a period not exceeding six months.
32(2) The competent authority shall publish any temporary prohibition on its
official website within 5 business days of the decision.
"""

result = run(REGULATION_TEXT)

print("=" * 65)
print(f"COMPLIANCE SUMMARY | {result.regulation_name}")
print("=" * 65)
print(f"Jurisdiction : {result.jurisdiction}")
if result.in_force_date:
    print(f"In force     : {result.in_force_date}")

print(f"\nOBLIGATIONS ({len(result.obligations)}):")
for o in result.obligations:
    status = "ongoing" if o.is_ongoing else "one-off"
    print(f"\n  [{o.source_article}] ({status})")
    print(f"  {o.obligation}")
    print(f"  Applies to : {o.applies_to}")
    if o.deadline:
        print(f"  Deadline   : {o.deadline}")

print(f"\nKEY DEADLINES ({len(result.key_deadlines)}):")
for d in result.key_deadlines:
    print(f"  - {d}")

print(f"\nPENALTIES ({len(result.penalties)}):")
for p in result.penalties:
    print(f"\n  [{p.source_article}]")
    print(f"  Trigger  : {p.trigger}")
    if p.maximum_fine:
        print(f"  Max fine : {p.maximum_fine}")
    if p.other_consequences:
        for c in p.other_consequences:
            print(f"  Other    : {c}")

if result.high_priority_gaps:
    print(f"\nHIGH-PRIORITY GAPS ({len(result.high_priority_gaps)}):")
    for g in result.high_priority_gaps:
        print(f"  - {g}")

## Use your own data

Replace `REGULATION_TEXT` above with:
- **The text of any regulation, directive, or statutory instrument** — paste it in as a plain string
- **A specific chapter or part**, if you only need to analyse a subset of a longer document

The agent will return every obligation, deadline, and penalty it can find — each one tied to the exact article in your text, flagged as ongoing or one-off, with a list of the gaps compliance teams most commonly miss.